# EP03 — Performance Metrics & Risk-Adjusted Returns
**Quantifaya · Classical Quantitative Finance Series · Episode 3**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Godwin-88/quantifire-web/blob/main/public/notebooks/ep03-performance-metrics.ipynb)

> **Learning objective:** Understand why the Sharpe Ratio is broken for asymmetric strategies, derive the Sortino fix, and build a complete performance attribution framework.

**Companion post:** [quantifaya.com/blog/ep03-sharpe-vs-sortino-which-metric](https://quantifaya.com/blog/ep03-sharpe-vs-sortino-which-metric)

---
*Quantifaya research notebooks are provided for educational purposes only. Nothing here constitutes financial advice.*

## Learning Objectives

By the end of this notebook you will be able to:

- **Compute** Sharpe, Sortino, and Calmar ratios from return series
- **Explain** why Sharpe penalizes upside volatility equally to downside
- **Derive** downside deviation for the Sortino Ratio
- **Analyze** return distribution shape: skewness, kurtosis, VaR, CVaR
- **Track** rolling performance metrics to detect regime changes
- **Identify** the "Sharpe Ratio trap" — strategies that manufacture high Sharpe by selling tail risk
- **Build** a complete performance attribution report

## Mathematical Prerequisites

### Sharpe Ratio

$$S = \frac{\bar{r}_p - r_f}{\sigma_p}$$

Where $\sigma_p$ is total volatility (both upside and downside).

### Sortino Ratio

$$\text{Sortino} = \frac{\bar{r}_p - r_f}{\sigma_{\text{downside}}}$$

Where downside deviation is:

$$\sigma_{\text{downside}} = \sqrt{\frac{1}{T} \sum_{t=1}^{T} \min(r_t - r_{\text{target}}, 0)^2}$$

### Calmar Ratio

$$\text{Calmar} = \frac{\text{CAGR}}{\text{Max Drawdown}}$$

### Distribution Moments

**Skewness:** Measures asymmetry. Positive = right-skewed (large gains), Negative = left-skewed (large losses).

$$\text{Skew} = \frac{\mathbb{E}[(r - \mu)^3]}{\sigma^3}$$

**Excess Kurtosis:** Measures fat-tailedness. >0 means fatter tails than normal.

$$\text{Kurt}_{\text{excess}} = \frac{\mathbb{E}[(r - \mu)^4]}{\sigma^4} - 3$$

## Setup: Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries loaded successfully.")

## 1. Performance Metric Functions

Core implementations of Sharpe, Sortino, Calmar, and tail risk measures:

In [ ]:
def sharpe_ratio(returns, rf=0.03, periods_per_year=252):
    """
    Calculate annualized Sharpe Ratio.
    
    Args:
        returns: Array of periodic returns (e.g., daily)
        rf: Annual risk-free rate
        periods_per_year: Number of periods per year (252 for daily)
    
    Returns:
        Annualized Sharpe Ratio
    """
    excess_returns = returns - rf / periods_per_year
    return np.sqrt(periods_per_year) * excess_returns.mean() / excess_returns.std()


def sortino_ratio(returns, rf=0.03, periods_per_year=252, target=0.0):
    """
    Calculate annualized Sortino Ratio.
    
    Args:
        returns: Array of periodic returns
        rf: Annual risk-free rate
        periods_per_year: Number of periods per year
        target: Target return (default 0, can use rf/periods_per_year)
    
    Returns:
        Annualized Sortino Ratio
    """
    excess_returns = returns - rf / periods_per_year
    downside_returns = excess_returns[excess_returns < target]
    
    if len(downside_returns) == 0:
        return np.inf  # No downside deviation
    
    downside_deviation = np.sqrt(np.mean(downside_returns ** 2))
    return np.sqrt(periods_per_year) * excess_returns.mean() / downside_deviation


def calmar_ratio(returns, periods_per_year=252):
    """
    Calculate Calmar Ratio: CAGR / Max Drawdown.
    """
    # CAGR
    cumulative = (1 + returns).cumprod()
    n_years = len(returns) / periods_per_year
    cagr = cumulative.iloc[-1] ** (1 / n_years) - 1 if hasattr(cumulative, 'iloc') else cumulative[-1] ** (1 / n_years) - 1
    
    # Max Drawdown
    rolling_max = cumulative.cummax() if hasattr(cumulative, 'cummax') else np.maximum.accumulate(cumulative)
    drawdowns = cumulative / rolling_max - 1
    max_dd = abs(drawdowns.min())
    
    return cagr / max_dd if max_dd > 0 else np.inf


def max_drawdown(returns, periods_per_year=252):
    """Calculate maximum drawdown."""
    cumulative = (1 + returns).cumprod() if hasattr(returns, 'cumprod') else np.cumprod(1 + returns)
    rolling_max = cumulative.cummax() if hasattr(cumulative, 'cummax') else np.maximum.accumulate(cumulative)
    drawdowns = cumulative / rolling_max - 1
    return abs(drawdowns.min())


def expected_shortfall(returns, alpha=0.05):
    """
    Expected Shortfall (CVaR): expected loss given VaR is breached.
    """
    var = -np.percentile(returns, alpha * 100)
    tail_losses = returns[returns <= -var]
    return -tail_losses.mean() if len(tail_losses) > 0 else var


def annualized_return(returns, periods_per_year=252):
    """Calculate annualized return."""
    n_years = len(returns) / periods_per_year
    cumulative = (1 + returns).prod() if hasattr(returns, 'prod') else np.prod(1 + returns)
    return cumulative ** (1 / n_years) - 1


def annualized_volatility(returns, periods_per_year=252):
    """Calculate annualized volatility."""
    return returns.std() * np.sqrt(periods_per_year)


print("Performance metric functions defined.")

## 2. Complete Performance Attribution Report

A single function that computes all key metrics:

In [ ]:
def complete_performance_attribution(returns, rf=0.03, periods_per_year=252, name="Strategy"):
    """
    Full performance attribution report.
    """
    excess = returns - rf / periods_per_year
    
    # Basic stats
    ann_return = annualized_return(returns, periods_per_year)
    ann_vol = annualized_volatility(returns, periods_per_year)
    
    # Risk metrics
    sharpe = sharpe_ratio(returns, rf, periods_per_year)
    sortino = sortino_ratio(returns, rf, periods_per_year)
    calmar = calmar_ratio(returns, periods_per_year)
    
    # Distribution moments
    skew = stats.skew(returns)
    kurt = stats.kurtosis(returns)  # Excess kurtosis
    
    # Drawdown
    max_dd = max_drawdown(returns, periods_per_year)
    
    # VaR and CVaR
    var_95 = -np.percentile(returns, 5)
    cvar_95 = expected_shortfall(returns, 0.05)
    
    report = {
        'Strategy': name,
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Sortino/Sharpe': sortino / sharpe if sharpe > 0 else np.nan,
        'Calmar Ratio': calmar,
        'Max Drawdown': max_dd,
        'Skewness': skew,
        'Excess Kurtosis': kurt,
        '95% VaR (daily)': var_95,
        '95% CVaR (daily)': cvar_95,
    }
    
    return pd.Series(report)


def print_formatted_report(report):
    """Pretty-print a performance report."""
    print(f"\n{'='*50}")
    print(f"Strategy: {report['Strategy']}")
    print(f"{'='*50}")
    print(f"  Annual Return:      {report['Annual Return']:>8.2%}")
    print(f"  Annual Volatility:  {report['Annual Volatility']:>8.2%}")
    print(f"  Sharpe Ratio:       {report['Sharpe Ratio']:>8.3f}")
    print(f"  Sortino Ratio:      {report['Sortino Ratio']:>8.3f}")
    sort_sharpe = report['Sortino/Sharpe']
    print(f"  Sortino/Sharpe:     {sort_sharpe:>8.2f}")
    print(f"  Calmar Ratio:       {report['Calmar Ratio']:>8.3f}")
    print(f"  Max Drawdown:       {report['Max Drawdown']:>8.2%}")
    print(f"  Skewness:           {report['Skewness']:>8.3f}")
    print(f"  Excess Kurtosis:    {report['Excess Kurtosis']:>8.3f}")
    print(f"  95% VaR (daily):    {report['95% VaR (daily)']:>8.2%}")
    print(f"  95% CVaR (daily):   {report['95% CVaR (daily)']:>8.2%}")

## 3. Simulate Three Strategy Archetypes

We simulate three distinct return distributions to illustrate the Sharpe vs Sortino difference:

In [ ]:
np.random.seed(42)
n_days = 252 * 3  # 3 years

# Strategy A: Symmetric returns (normal distribution)
returns_A = np.random.normal(0.0005, 0.01, n_days)

# Strategy B: Positive skew (small losses, occasional big wins)
# Like momentum strategies or trend following
returns_B = np.random.normal(-0.0002, 0.008, n_days)
gain_days = np.random.choice(n_days, size=15, replace=False)
returns_B[gain_days] += np.random.uniform(0.04, 0.08, 15)

# Strategy C: Negative skew (small gains, occasional big losses)
# Like selling options / tail risk selling
returns_C = np.random.normal(0.0008, 0.006, n_days)
loss_days = np.random.choice(n_days, size=5, replace=False)
returns_C[loss_days] -= np.random.uniform(0.08, 0.15, 5)

df_returns = pd.DataFrame({
    'Strategy_A_Symmetric': returns_A,
    'Strategy_B_PosSkew': returns_B,
    'Strategy_C_NegSkew': returns_C,
})

# Convert to Series for our functions
s_A = pd.Series(returns_A)
s_B = pd.Series(returns_B)
s_C = pd.Series(returns_C)

print("=== Three Strategy Archetypes ===")
print(f"Strategy A (Symmetric):  {len(s_A)} daily returns")
print(f"Strategy B (Pos Skew):   {len(s_B)} daily returns")
print(f"Strategy C (Neg Skew):   {len(s_C)} daily returns")

# Plot return distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, s, title in zip(axes, [s_A, s_B, s_C], 
                         ['Symmetric (Normal)', 'Positive Skew', 'Negative Skew']):
    ax.hist(s * 100, bins=50, alpha=0.7, edgecolor='black', color='steelblue')
    ax.axvline(s.mean() * 100, color='red', linestyle='--', linewidth=2, label=f'Mean: {s.mean()*100:.3f}%')
    ax.set_xlabel('Daily Return (%)')
    ax.set_ylabel('Frequency')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Performance Reports for All Three Strategies

In [ ]:
print("\n" + "="*60)
print("PERFORMANCE ATTRIBUTION REPORTS")
print("="*60)

report_A = complete_performance_attribution(s_A, name="Strategy A: Symmetric")
report_B = complete_performance_attribution(s_B, name="Strategy B: Positive Skew")
report_C = complete_performance_attribution(s_C, name="Strategy C: Negative Skew")

print_formatted_report(report_A)
print_formatted_report(report_B)
print_formatted_report(report_C)

### Key Observations

| Strategy | Sharpe | Sortino | Sortino/Sharpe | Skewness | Interpretation |
|----------|--------|---------|----------------|----------|----------------|
| A (Symmetric) | ~0.6 | ~0.9 | ~1.45 | ~0 | Balanced upside/downside |
| B (Pos Skew) | ~0.8 | ~1.5 | ~1.83 | >0 | Large gains drive volatility — **desirable** |
| C (Neg Skew) | ~0.8 | ~0.6 | ~0.82 | <0 | Large losses drive volatility — **dangerous** |

**The Sortino/Sharpe ratio is the key signal:**
- **> 1.5**: Positive skew (desirable asymmetry)
- **≈ 1.0**: Symmetric (neutral)
- **< 0.8**: Negative skew (hidden tail risk)

## 5. The Sharpe Ratio Trap

Demonstrate how negative-skew strategies can manufacture deceptively high Sharpe ratios:

In [ ]:
# Simulate a "Sharpe trap" strategy: consistent small gains, rare catastrophic loss
np.random.seed(123)
n_days = 252 * 5  # 5 years

# Base returns: smooth, consistent small gains
trap_returns = np.random.normal(0.0008, 0.003, n_days)

# One catastrophic event at year 4
crash_day = 252 * 3 + 60  # Day ~800
trap_returns[crash_day] = -0.40  # 40% single-day loss

s_trap = pd.Series(trap_returns)

# Compute metrics BEFORE the crash (first 3 years)
s_trap_pre = s_trap.iloc[:252*3]
report_pre = complete_performance_attribution(s_trap_pre, name="Trap Strategy (Pre-Crash)")

# Compute metrics AFTER the crash (full period)
report_post = complete_performance_attribution(s_trap, name="Trap Strategy (Post-Crash)")

print("\n" + "="*60)
print("THE SHARPE RATIO TRAP")
print("="*60)

print_formatted_report(report_pre)
print("\n>>> This looks great! Sharpe = {:.2f}, Sortino = {:.2f}".format(
    report_pre['Sharpe Ratio'], report_pre['Sortino Ratio']))
print(">>> But notice: Sortino < Sharpe → negative skew warning!")

print_formatted_report(report_post)
print("\n>>> After the crash: all metrics collapse.")
print(">>> The high Sharpe was manufactured by selling tail risk.")

# Plot cumulative returns
fig, ax = plt.subplots(figsize=(14, 6))

cumulative = (1 + s_trap).cumprod()
ax.plot(cumulative.index, cumulative.values, linewidth=2, color='steelblue', label='Cumulative Return')

# Mark the crash
ax.axvline(x=crash_day, color='red', linestyle='--', linewidth=2, label='Crash Day (-40%)')
ax.axvspan(0, 252*3, alpha=0.1, color='green', label='Pre-Crash Period')

ax.set_xlabel('Trading Day')
ax.set_ylabel('Cumulative Return')
ax.set_title('The Sharpe Ratio Trap: Smooth Returns Until Catastrophe', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Rolling Metrics: Detecting Regime Changes

Performance metrics drift over time. Rolling windows help detect regime shifts:

In [ ]:
def rolling_sharpe(returns, window=63, rf=0.03, periods_per_year=252):
    """Rolling Sharpe Ratio over a window."""
    excess = returns - rf / periods_per_year
    rolling_mean = excess.rolling(window).mean()
    rolling_std = excess.rolling(window).std()
    return np.sqrt(periods_per_year) * rolling_mean / rolling_std


def rolling_sortino(returns, window=63, rf=0.03, periods_per_year=252):
    """Rolling Sortino Ratio over a window."""
    excess = returns - rf / periods_per_year
    
    def sortino_window(win):
        downside = win[win < 0]
        if len(downside) == 0:
            return np.nan
        dd = np.sqrt(np.mean(downside ** 2))
        return np.sqrt(periods_per_year) * win.mean() / dd
    
    return excess.rolling(window).apply(sortino_window, raw=False)


# Compute rolling metrics for Strategy C (negative skew)
rolling_s = rolling_sharpe(s_C, window=63)
rolling_so = rolling_sortino(s_C, window=63)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Rolling Sharpe vs Sortino
axes[0].plot(rolling_s.index, rolling_s.values, label='Rolling Sharpe (63d)', 
             color='steelblue', linewidth=2)
axes[0].plot(rolling_so.index, rolling_so.values, label='Rolling Sortino (63d)', 
             color='orange', linewidth=2)
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.5)
axes[0].axhline(1.0, color='green', linestyle=':', linewidth=1, alpha=0.5, label='Sharpe = 1.0')
axes[0].set_ylabel('Ratio')
axes[0].set_title('Rolling Sharpe vs Sortino - Detect Regime Changes', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Sortino/Sharpe ratio over time
sort_sharpe_ratio = rolling_so / rolling_s
axes[1].plot(sort_sharpe_ratio.index, sort_sharpe_ratio.values, 
             color='purple', linewidth=2)
axes[1].axhline(1.0, color='black', linestyle='--', linewidth=1, label='Sortino = Sharpe')
axes[1].axhline(0.8, color='red', linestyle=':', linewidth=1, alpha=0.5, label='Danger Zone (<0.8)')
axes[1].axhline(1.5, color='green', linestyle=':', linewidth=1, alpha=0.5, label='Desirable Zone (>1.5)')
axes[1].set_xlabel('Trading Day')
axes[1].set_ylabel('Sortino / Sharpe')
axes[1].set_title('Sortino/Sharpe Ratio: Early Warning for Skew Changes', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Rolling Metric Signals ===")
print("Watch for:")
print("  • Sharpe & Sortino converging → distribution becoming symmetric")
print("  • Sortino dropping below Sharpe → negative skew emerging (DANGER)")
print("  • Both metrics declining together → alpha decaying")

## 7. DeFi-Specific Performance Analysis

Analyze strategies with return distributions typical of DeFi protocols:

In [ ]:
# Simulate DeFi strategy returns
np.random.seed(200)
n_days = 252 * 2  # 2 years

# Liquidity Provision: steady fees + impermanent loss risk
defi_lp = np.random.normal(0.001, 0.005, n_days)
il_events = np.random.choice(n_days, size=3, replace=False)
defi_lp[il_events] -= np.random.uniform(0.05, 0.12, 3)  # IL events

# Yield Farming: high yields + incentive cliff risk
defi_yield = np.random.normal(0.002, 0.004, n_days)
cliff_day = 252 * 1.5  # Incentives end at 18 months
defi_yield[int(cliff_day):] = np.random.normal(0.0003, 0.008, n_days - int(cliff_day))

# Funding Rate Arb: relatively symmetric
defi_funding = np.random.normal(0.0006, 0.003, n_days)

s_lp = pd.Series(defi_lp)
s_yield = pd.Series(defi_yield)
s_funding = pd.Series(defi_funding)

print("\n" + "="*60)
print("DeFi STRATEGY PERFORMANCE COMPARISON")
print("="*60)

for s, name in [(s_lp, "Liquidity Provision (LP)"), 
                 (s_yield, "Yield Farming"), 
                 (s_funding, "Funding Rate Arbitrage")]:
    r = complete_performance_attribution(s, name=name)
    print_formatted_report(r)
    print()

# Summary comparison
defi_comparison = pd.DataFrame([
    complete_performance_attribution(s_lp, name="LP"),
    complete_performance_attribution(s_yield, name="Yield Farming"),
    complete_performance_attribution(s_funding, name="Funding Arb"),
]).set_index('Strategy')[['Sharpe Ratio', 'Sortino Ratio', 'Sortino/Sharpe', 'Skewness', 'Max Drawdown']]

print("\n=== DeFi Strategy Summary ===")
print(defi_comparison.to_string(formatters={
    'Sharpe Ratio': '{:.3f}'.format,
    'Sortino Ratio': '{:.3f}'.format,
    'Sortino/Sharpe': '{:.2f}'.format,
    'Skewness': '{:.3f}'.format,
    'Max Drawdown': '{:.2%}'.format,
}))

## 8. Minimum Reporting Standard

Never evaluate a strategy with a single metric. Always report:

In [ ]:
def minimum_reporting_standard(returns, name="Strategy", rf=0.03):
    """
    The minimum set of metrics every strategy should report.
    """
    report = complete_performance_attribution(returns, rf, name=name)
    
    print(f"\n{'='*55}")
    print(f"  MINIMUM REPORTING STANDARD: {name}")
    print(f"{'='*55}")
    print(f"  1. Sharpe Ratio:        {report['Sharpe Ratio']:.3f}")
    print(f"  2. Sortino Ratio:       {report['Sortino Ratio']:.3f}")
    print(f"  3. Max Drawdown:        {report['Max Drawdown']:.2%}")
    print(f"  4. Skewness:            {report['Skewness']:.3f}")
    print(f"  5. Excess Kurtosis:     {report['Excess Kurtosis']:.3f}")
    print(f"  6. 95% CVaR (daily):    {report['95% CVaR (daily)']:.2%}")
    print(f"  ─" * 20)
    
    # Red flags
    flags = []
    if report['Sortino/Sharpe'] < 0.8:
        flags.append("⚠️  Sortino < Sharpe → negative skew, hidden tail risk")
    if report['Excess Kurtosis'] > 3:
        flags.append("⚠️  High kurtosis → fat tails, extreme events likely")
    if report['Max Drawdown'] > 0.30:
        flags.append("⚠️  Max DD > 30% → severe drawdown risk")
    if report['Skewness'] < -1:
        flags.append("⚠️  Strong negative skew → asymmetric downside")
    
    if flags:
        print("\n  RED FLAGS:")
        for flag in flags:
            print(f"  {flag}")
    else:
        print("\n  ✅ No major red flags detected.")
    
    print(f"{'='*55}")
    return report

# Apply minimum reporting standard to all strategies
for s, name in [(s_A, "Symmetric"), (s_B, "Positive Skew"), 
                 (s_C, "Negative Skew"), (s_trap, "Sharpe Trap")]:
    minimum_reporting_standard(s, name=name)

## Key Takeaways

1. **Sharpe penalizes upside volatility** — it treats good surprises the same as bad surprises.

2. **Sortino uses only downside deviation** — it answers: how much return per unit of *bad* risk?

3. **The Sortino/Sharpe ratio is informative:**
   - **> 1.5**: Positive skew (desirable)
   - **≈ 1.0**: Symmetric (neutral)
   - **< 0.8**: Negative skew (dangerous)

4. **Never use a single metric.** Minimum reporting: Sharpe + Sortino + Max DD + Skewness + CVaR.

5. **Track rolling metrics.** Regime changes show up as divergences between Sharpe and Sortino before they show up in P&L.

---

**References:**

- Sharpe, W.F. (1966). "Mutual Fund Performance." *Journal of Business*, 39(S1), 119–138.
- Sortino, F.A. & Price, L.N. (1994). "Performance Measurement in a Downside Risk Framework." *JOI*, 3(3), 50–58.
- Taleb, N.N. (2007). *The Black Swan*. Random House.

---
*Quantifaya — Quantitative Finance for Web2 & Web3. Not financial advice.*